In [ ]:
# This public repository version reads input files from the local repository.


Α.

In [ ]:
from pathlib import Path
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "outputs" / "oof" / "results").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open Jupyter from this repository, with outputs/oof/results available")
OOF_RESULTS_DIR = REPO_ROOT / "outputs" / "oof" / "results"
OOF_PLOT_DIR = REPO_ROOT / "outputs" / "oof" / "regenerated_figures"
OOF_PLOT_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# INPUT FILES
# ============================================================
MIXED_PATH = OOF_RESULTS_DIR / "mixed_GHPI_REPEATED20x5_FINAL_all_OOF_scores.csv"
REAL_PATH = OOF_RESULTS_DIR / "real_only_GHPI_REPEATED20x5_FINAL_all_OOF_scores.csv"

# ============================================================
# LOAD
# ============================================================
mixed_df = pd.read_csv(MIXED_PATH)
real_df  = pd.read_csv(REAL_PATH)

# ============================================================
# FUNCTION: COMPUTE PRESSURE-ASSIGNMENT STABILITY
# ============================================================
def compute_assignment_stability(df, subset_name):
    temp = df.copy()

    # High-pressure assignment for each repetition
    temp["is_high_pressure"] = (
        temp["GHPI_OOF_repeat"] > temp["GHPI_OOF_trainP75_repeat"]
    ).astype(int)

    # Aggregate across repetitions for each citation context
    out = (
        temp.groupby("original_index", as_index=False)
            .agg(
                n_repetitions=("is_high_pressure", "size"),
                n_high=("is_high_pressure", "sum"),
                median_GHPI=("GHPI_OOF_repeat", "median"),
                median_trainP75=("GHPI_OOF_trainP75_repeat", "median")
            )
    )

    out["p_high"] = out["n_high"] / out["n_repetitions"]
    out["subset"] = subset_name

    return out

mixed_stability = compute_assignment_stability(mixed_df, "Mixed")
real_stability  = compute_assignment_stability(real_df, "Real-only")

plot_df = pd.concat([mixed_stability, real_stability], ignore_index=True)

# ============================================================
# QUICK CHECKS
# ============================================================
print("Mixed unique citation contexts:", mixed_stability.shape[0])
print("Real-only unique citation contexts:", real_stability.shape[0])

print("\nMixed repetitions per citation:")
print(mixed_stability["n_repetitions"].value_counts().sort_index())

print("\nReal-only repetitions per citation:")
print(real_stability["n_repetitions"].value_counts().sort_index())

# ============================================================
# PAPER-READY STYLE
# ============================================================
sns.set_theme(style="whitegrid", context="paper", font_scale=1.0)

plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 300,
    "savefig.dpi": 300
})

# ============================================================
# FIGURE WITH TWO PANELS
# ============================================================
bins = np.linspace(0, 1, 21)

fig, axes = plt.subplots(1, 2, figsize=(6, 4), sharey=True)

subsets = ["Mixed", "Real-only"]

for ax, subset in zip(axes, subsets):
    sub_df = plot_df[plot_df["subset"] == subset]

    sns.histplot(
        data=sub_df,
        x="p_high",
        bins=bins,
        stat="density",
        element="step",
        fill=False,
        linewidth=1.8,
        ax=ax
    )

    sns.kdeplot(
        data=sub_df,
        x="p_high",
        linewidth=1.6,
        ax=ax
    )

    ax.set_title(subset)
    ax.set_xlabel(r"$P(\mathrm{High})$")
    ax.set_xlim(0, 1)
    ax.set_xticks(np.linspace(0, 1, 6))
    ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.5)
    ax.grid(axis="x", visible=False)
    sns.despine(ax=ax)

axes[0].set_ylabel("Density")
axes[1].set_ylabel("")

fig.supxlabel(
    r"Proportion of OOF repetitions in which a citation context is assigned to High pressure",
    y=0.04
)

plt.tight_layout(rect=[0, 0.05, 1, 1])

# ============================================================
# SAVE
# ============================================================
plt.savefig(OOF_PLOT_DIR / "OOF_pressure_assignment_stability_2panels.pdf", bbox_inches="tight", dpi=300)
#plt.savefig("OOF_pressure_assignment_stability_2panels.png", bbox_inches="tight", dpi=600)

plt.show()

Β. Αυτή θα ήταν η τελική έκδοση χωρίς KDE, μόνο με histogram, σε δύο panels. Επειδή το P(High) παίρνει διακριτές τιμές ανά 0.05, αυτή η έκδοση είναι και πιο πιστή στα δεδομένα.

In [ ]:
from pathlib import Path
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "outputs" / "oof" / "results").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open Jupyter from this repository, with outputs/oof/results available")
OOF_RESULTS_DIR = REPO_ROOT / "outputs" / "oof" / "results"
OOF_PLOT_DIR = REPO_ROOT / "outputs" / "oof" / "regenerated_figures"
OOF_PLOT_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# INPUT FILES
# ============================================================
MIXED_PATH = OOF_RESULTS_DIR / "mixed_GHPI_REPEATED20x5_FINAL_all_OOF_scores.csv"
REAL_PATH = OOF_RESULTS_DIR / "real_only_GHPI_REPEATED20x5_FINAL_all_OOF_scores.csv"

# ============================================================
# LOAD DATA
# ============================================================
mixed_df = pd.read_csv(MIXED_PATH)
real_df  = pd.read_csv(REAL_PATH)

# ============================================================
# COMPUTE PRESSURE-ASSIGNMENT STABILITY
# ============================================================
def compute_assignment_stability(df, subset_name):
    temp = df.copy()

    # High-pressure assignment within each OOF repetition
    temp["is_high_pressure"] = (
        temp["GHPI_OOF_repeat"] >
        temp["GHPI_OOF_trainP75_repeat"]
    ).astype(int)

    # Aggregate the 20 assignments for each citation context
    stability = (
        temp.groupby("original_index", as_index=False)
            .agg(
                n_repetitions=("is_high_pressure", "size"),
                n_high=("is_high_pressure", "sum")
            )
    )

    # Proportion of repetitions assigned to High pressure
    stability["p_high"] = (
        stability["n_high"] / stability["n_repetitions"]
    )

    stability["subset"] = subset_name

    return stability


mixed_stability = compute_assignment_stability(
    mixed_df,
    "Mixed"
)

real_stability = compute_assignment_stability(
    real_df,
    "Real-only"
)

# ============================================================
# SANITY CHECKS
# ============================================================
print("Mixed citation contexts:", len(mixed_stability))
print("Real-only citation contexts:", len(real_stability))

print("\nMixed repetitions per citation:")
print(mixed_stability["n_repetitions"].value_counts().sort_index())

print("\nReal-only repetitions per citation:")
print(real_stability["n_repetitions"].value_counts().sort_index())

# ============================================================
# PAPER-READY STYLE
# ============================================================
sns.set_theme(
    style="whitegrid",
    context="paper",
    font_scale=1.0
)

plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 300,
    "savefig.dpi": 300
})

# ============================================================
# HISTOGRAM BINS
# ============================================================
# 20 repetitions -> possible P(High):
# 0.00, 0.05, 0.10, ..., 1.00

step = 0.05

# Bin edges centered around each possible discrete value
bins = np.arange(
    -step / 2,
    1 + step,
    step
)

# ============================================================
# TWO-PANEL FIGURE
# ============================================================
fig, axes = plt.subplots(
    1,
    2,
    figsize=(6, 4),
    sharey=True
)

datasets = [
    (mixed_stability, "Mixed"),
    (real_stability, "Real-only")
]

for ax, (data, title) in zip(axes, datasets):

    sns.histplot(
        data=data,
        x="p_high",
        bins=bins,
        stat="count",
        discrete=False,
        edgecolor="black",
        linewidth=0.6,
        ax=ax
    )

    # Panel title
    ax.set_title(title)

    # X-axis
    ax.set_xlim(-0.025, 1.025)
    ax.set_xticks(
        [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    )

    ax.set_xlabel(
        r"$P(\mathrm{High})$"
    )

    # Grid only on y-axis
    ax.grid(
        axis="y",
        linestyle="--",
        linewidth=0.6,
        alpha=0.5
    )

    ax.grid(
        axis="x",
        visible=False
    )

    sns.despine(ax=ax)

# Y labels
axes[0].set_ylabel("Number of citation contexts")
axes[1].set_ylabel("")

# Shared explanatory x-label
fig.supxlabel(
    "Proportion of OOF repetitions assigned to High pressure",
    y=0.035
)

plt.tight_layout(
    rect=[0, 0.055, 1, 1]
)

# ============================================================
# SAVE
# ============================================================
plt.savefig(
    OOF_PLOT_DIR / "OOF_pressure_assignment_stability_2panels_without_KDE.pdf",
    bbox_inches="tight"
)

plt.show()

Γ. Παρουσίαση η οποία είναι πολύ πιο καθαρή για appendix

In [ ]:
from pathlib import Path
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "outputs" / "oof" / "results").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open Jupyter from this repository, with outputs/oof/results available")
OOF_RESULTS_DIR = REPO_ROOT / "outputs" / "oof" / "results"
OOF_PLOT_DIR = REPO_ROOT / "outputs" / "oof" / "regenerated_figures"
OOF_PLOT_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# INPUT FILES
# ============================================================
MIXED_PATH = OOF_RESULTS_DIR / "mixed_GHPI_REPEATED20x5_FINAL_all_OOF_scores.csv"
REAL_PATH = OOF_RESULTS_DIR / "real_only_GHPI_REPEATED20x5_FINAL_all_OOF_scores.csv"

# ============================================================
# LOAD DATA
# ============================================================
mixed_df = pd.read_csv(MIXED_PATH)
real_df  = pd.read_csv(REAL_PATH)

# ============================================================
# COMPUTE P(HIGH) FOR EACH CITATION CONTEXT
# ============================================================
def compute_assignment_stability(df, subset_name):
    temp = df.copy()

    # High-pressure assignment in each repetition
    temp["is_high_pressure"] = (
        temp["GHPI_OOF_repeat"] > temp["GHPI_OOF_trainP75_repeat"]
    ).astype(int)

    # Aggregate over repetitions for each citation context
    stability = (
        temp.groupby("original_index", as_index=False)
            .agg(
                n_repetitions=("is_high_pressure", "size"),
                n_high=("is_high_pressure", "sum")
            )
    )

    stability["p_high"] = stability["n_high"] / stability["n_repetitions"]
    stability["subset"] = subset_name

    return stability

mixed_stability = compute_assignment_stability(mixed_df, "Mixed")
real_stability  = compute_assignment_stability(real_df, "Real-only")

# ============================================================
# CLASSIFY INTO 3 STABILITY GROUPS
# ============================================================
def categorize_stability(p):
    if p <= 0.10:
        return "Stable Low"
    elif p >= 0.90:
        return "Stable High"
    else:
        return "Variable"

plot_df = pd.concat([mixed_stability, real_stability], ignore_index=True)
plot_df["stability_group"] = plot_df["p_high"].apply(categorize_stability)

# ============================================================
# AGGREGATE COUNTS + PERCENTAGES
# ============================================================
order = ["Stable Low", "Variable", "Stable High"]

summary = (
    plot_df.groupby(["subset", "stability_group"])
           .size()
           .reset_index(name="count")
)

# ensure all category combinations exist
all_combinations = pd.MultiIndex.from_product(
    [["Mixed", "Real-only"], order],
    names=["subset", "stability_group"]
).to_frame(index=False)

summary = all_combinations.merge(
    summary,
    on=["subset", "stability_group"],
    how="left"
)

summary["count"] = summary["count"].fillna(0).astype(int)

totals = summary.groupby("subset")["count"].transform("sum")
summary["percent"] = 100 * summary["count"] / totals

print(summary)

# ============================================================
# PAPER-READY STYLE
# ============================================================
sns.set_theme(style="whitegrid", context="paper", font_scale=1.0)

plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 300,
    "savefig.dpi": 300
})

# ============================================================
# TWO-PANEL BARPLOT
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(6, 4), sharey=True)

subsets = ["Mixed", "Real-only"]

for ax, subset in zip(axes, subsets):
    sub_df = summary[summary["subset"] == subset].copy()
    sub_df["stability_group"] = pd.Categorical(
        sub_df["stability_group"],
        categories=order,
        ordered=True
    )
    sub_df = sub_df.sort_values("stability_group")

    sns.barplot(
        data=sub_df,
        x="stability_group",
        y="percent",
        ax=ax
    )

    ax.set_title(subset)
    ax.set_xlabel("")
    ax.set_ylabel("Percentage of citation contexts" if subset == "Mixed" else "")
    ax.set_ylim(0, max(summary["percent"].max() + 8, 100))

    ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.5)
    ax.grid(axis="x", visible=False)
    sns.despine(ax=ax)

    # annotate bars with count and percentage
    for i, row in enumerate(sub_df.itertuples()):
        ax.text(
            i,
            row.percent + 1.0,
            f"{row.count}\n({row.percent:.1f}%)",
            ha="center",
            va="bottom",
            fontsize=9
        )

#fig.supxlabel("Pressure-assignment stability category", y=0.04)

plt.tight_layout(rect=[0, 0.05, 1, 1])

# ============================================================
# SAVE
# ============================================================
plt.savefig(OOF_PLOT_DIR / "OOF_pressure_assignment_stability_categories.pdf", bbox_inches="tight", dpi=300)
#plt.savefig("OOF_pressure_assignment_stability_categories.png", bbox_inches="tight", dpi=600)

plt.show()